## Building a Smarter Chain-of-Thought AI with Python

Standard LLM queries often struggle with complex logic, failing in a single "leap" of intuition. This Python implementation solves that by enforcing a **structured 4-step reasoning pipeline** powered by **Ollama** however you can use any LLM endpoints.

### The Core Logic
The engine breaks down a problem into four distinct phases:
1.  **Direct Solve:** Finding an initial answer.
2.  **Alternative Perspective:** Solving the problem again using a different method to ensure accuracy.
3.  **Validation:** A "referee" step that compares both previous results to identify discrepancies.
4.  **Final Synthesis:** Consolidating the verified logic into a concise final answer.

### Key Engineering Features
* **Parallel Execution:** To combat latency, we use `ThreadPoolExecutor` to run Step 1 and Step 2 simultaneously. This cuts response times by nearly 50% on 16GB RAM systems.
* **Robust Extraction:** Using regex and "last-line" fallbacks, the code ensures that even if the LLM is brief, the conclusion is never lost.
* **Feedback Loops:** Each subsequent step is injected with the context of the previous ones, creating a self-correcting AI workflow that mimics human critical thinking.

In [1]:
from clients.ollama import OllamaClient

In [2]:
llmclient=OllamaClient()

In [3]:
llmclient.check_health()

OLLAMA is Running


True

In [4]:
if llmclient.check_health():
    print("✅ Ollama connected successfully!\n")

OLLAMA is Running
✅ Ollama connected successfully!



In [5]:
from rag.cot import ChainOfThoughtReasoner

In [6]:
cot=ChainOfThoughtReasoner(llm_client=llmclient)

In [8]:
test_suite = [
    {
        "name": "The Simple Math Test",
        "problem": "John has 5 apples. He gives 2 to Mary, then buys 3 more. How many does he have?"
    },
    {
        "name": "The Temporal Paradox",
        "problem": "A plane leaves Tokyo (UTC+9) at 11 PM Tuesday. The flight to LA (UTC-8) takes 10 hours. What is the local time and day in LA upon landing?"
    },
    {
        "name": "The Inventory Logic Trap",
        "problem": "I have 5 'Green apples' and 5 'Red oranges'. If I remove all items named after a color that are NOT that color, how many items remain?"
    },
    {
        "name": "The Displacement Physics Riddle",
        "problem": "If you are in a boat and throw a heavy stone into the water, does the lake's water level rise or fall? Explain why."
    },
    {
        "name": "The Family Algebra Puzzle",
        "problem": "In a family, a girl has as many brothers as sisters, but a boy has twice as many sisters as brothers. How many boys and girls are there?"
    }
]

# 4. Execute Batch Processing
for test in test_suite[0:1]:
    print(f"\n🚀 TESTING: {test['name']}")
    print(f"📝 Problem: {test['problem']}")
    print("-" * 70)
    print("🤔 Multi-threaded Reasoning in progress...")
    
    # Execute the 4-step logic
    steps = cot.reason(test['problem'])
    
    # Print the formatted output
    print(cot.explain_reasoning())
    print("\n" + "="*70 + "\n")


🚀 TESTING: The Simple Math Test
📝 Problem: John has 5 apples. He gives 2 to Mary, then buys 3 more. How many does he have?
----------------------------------------------------------------------
🤔 Multi-threaded Reasoning in progress...
🚀 Running Methods A & B in parallel...

FINAL CHAIN OF THOUGHT REPORT
📍 STEP 1: Method A (Direct)
   Reasoning: John starts with 5 apples. After giving 2 to Mary, he has $5 - 2 = 3$ apples left. Then he buys 3 more apples, so he has $3 + 3 = 6$ apples. 

Result:...
   ✅ Conclusion: 6
------------------------------------------------------------
📍 STEP 2: Method B (Alternative)
   Reasoning: John starts with 5 apples. After giving 2 to Mary, he has 5 - 2 = 3 apples left. Then he buys 3 more, so 3 + 3 = 6. An alternative mental check: the n...
   ✅ Conclusion: 6
------------------------------------------------------------
📍 STEP 3: Validation
   Reasoning: No response from LLM....
   ✅ Conclusion: Error
-----------------------------------------------------

In [ ]:
# problem = ( "John has 5 apples. He gives 2 apples to Mary. "
#         "Then he buys 3 more apples. How many apples does John have now?" )
    
# print(f"\n📝 Problem:\n{problem}")
# print("\n" + "-" * 70)
# print("🤔 Reasoning with Chain-of-Thought...\n")

# # 5. Execute Reasoning
# steps = cot.reason(problem)

# # 6. Print Results
# print(cot.explain_reasoning())



📝 Problem:
John has 5 apples. He gives 2 apples to Mary. Then he buys 3 more apples. How many apples does John have now?

----------------------------------------------------------------------
🤔 Reasoning with Chain-of-Thought...

🚀 Running Methods A & B in parallel...

FINAL CHAIN OF THOUGHT REPORT
📍 STEP 1: Method A (Direct)
   Reasoning: John starts with 5 apples. After giving 2 to Mary, he has 5 - 2 = 3 apples. Then he buys 3 more apples, so 3 + 3 = 6. 

Result: 6...
   ✅ Conclusion: 6
------------------------------------------------------------
📍 STEP 2: Method B (Alternative)
   Reasoning: John starts with 5 apples. After giving 2 to Mary, he has 5 - 2 = 3 apples left. Then he buys 3 more apples, so 3 + 3 = 6. An alternative mental check...
   ✅ Conclusion: 6
------------------------------------------------------------
📍 STEP 3: Validation
   Reasoning: No response from LLM....
   ✅ Conclusion: Error
------------------------------------------------------------
📍 STEP 4: Final Ou

#### END ####